# 🔬 Quantitative TCN Trading Bot: SOL/USDT Optimization Suite
### Rigorous Empirical Research & Profit Maximization on SOL/USDT 4H Perpetual Futures (Alta Volatilità / High Beta (Calibrazione Robusta Anti-Overfitting))

Questo notebook addestra e ottimizza l'architettura **Temporal Convolutional Network (TCN)** per **SOL/USDT** con parametri quantitativi calibrati per il suo specifico profilo di volatilità:
- **Orizzonte di holding**: **6 barre 4H (24 ore)**
- **Barriere Volatili Dinamiche**: **$2.4 \times ATR$ Take Profit** e **$1.5 \times ATR$ Stop Loss** (minimo $3.5\%$)
- **Commissioni reali**: $0.05\%$ taker in ingresso $+ 0.05\%$ taker in uscita ($0.10\%$ round-trip)

Il notebook esegue in successione automatica i **3 esperimenti chiave**:
1. **Esperimento 1 — Grid Search su Validation Set:** Confronta 3 configurazioni di regolarizzazione (`learning_rate`, `dropout`, `weight_decay`) su orizzonte 6 barre con target fissi $+3.5\%$ TP e $-2.2\%$ SL.
2. **Esperimento 2 — Dynamic ATR Volatility Barriers:** Introduce barriere adattive basate sull'Average True Range ($2.4 \times ATR$ e $1.5 \times ATR$) con orizzonte 6 barre (24h).
3. **Esperimento 3 — Advanced Execution (Trailing Stop a Breakeven Progressivo + Compound Sizing 15%):** Sposta lo stop loss a breakeven in modo progressivo (dopo almeno $+2.5\%$ o al 50% del percorso verso TP), evitando di farsi fermare dal rumore naturale dei wick.
4. **Tabella Comparativa Finale & Champion Export:** Grafico multi-equity curve a confronto e salvataggio dei pesi del modello vincitore `tcn_sol_best_model.pth` e `scaler_sol.pkl`.


In [ ]:
# 1. Installazione dipendenze minime
!pip install ccxt pyyaml --quiet
print("Ambiente configurato con successo!")


In [ ]:
# 2. Import delle librerie e configurazione base
import os
import glob
import time
import json
import pickle
import logging
from datetime import datetime, timedelta, timezone
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, f1_score, precision_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

BASE_CONFIG = {
    "symbol": "SOL/USDT",
    "timeframe": "4h",
    "lookback_window": 64,
    "prediction_horizon": 6,       # 24 ore holding
    "taker_fee": 0.0005,           # 0.05% taker (0.10% round-trip)
    "embargo_bars": 12,            # 48 ore anti-leakage
    "batch_size": 64,
    "channels": [32, 32, 48, 48, 64], # ~65k parametri
    "kernel_size": 3,
    "max_epochs": 40,
    "patience": 10
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo di calcolo attivo: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU rilevata: {torch.cuda.get_device_name(0)}")


In [ ]:
# 3. Caricamento dati storici OHLCV (SOL/USDT)
def load_market_data():
    target_parquet = "SOL_USDT_4h.parquet"
    search_dirs = ["/kaggle/input", "data", "."]
    for d in search_dirs:
        if os.path.exists(d):
            for root, _, files in os.walk(d):
                for f in files:
                    if f.lower() == target_parquet.lower() or (f.lower().startswith("sol") and f.endswith(".parquet")):
                        file_path = os.path.join(root, f)
                        print(f"Caricamento dataset locale: {file_path}")
                        df = pd.read_parquet(file_path)
                        print(f"Caricate {len(df)} barre da {df['timestamp'].iloc[0]} a {df['timestamp'].iloc[-1]}")
                        return df

    # Auto-Download diretto da Binance Futures con paginazione (5 anni di storico)
    print("Download automatico dello storico completo da Binance Futures via CCXT...")
    import ccxt
    exchange = ccxt.binance({"enableRateLimit": True, "options": {"defaultType": "future"}})
    exchange.load_markets()
    symbol = BASE_CONFIG["symbol"]
    target_market = f"{symbol}:USDT" if f"{symbol}:USDT" in exchange.markets else symbol

    now = datetime.now(timezone.utc)
    start_dt = now - timedelta(days=1825) # 5 anni
    since_ms = int(start_dt.timestamp() * 1000)

    all_ohlcv = []
    limit = 1000
    while True:
        try:
            ohlcv = exchange.fetch_ohlcv(target_market, timeframe=BASE_CONFIG["timeframe"], since=since_ms, limit=limit)
            if not ohlcv or len(ohlcv) == 0:
                break
            all_ohlcv.extend(ohlcv)
            last_ts = ohlcv[-1][0]
            if last_ts == since_ms or len(ohlcv) < limit:
                break
            since_ms = last_ts + 1
            time.sleep(exchange.rateLimit / 1000.0)
        except Exception as e:
            print(f"Avviso durante download: {e}")
            break

    df = pd.DataFrame(all_ohlcv, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df.drop_duplicates(subset=["timestamp"], inplace=True)
    df.sort_values(by="timestamp", inplace=True)
    df.reset_index(drop=True, inplace=True)
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c])
    df["funding_rate"] = 0.0

    print(f"Download completato: {len(df)} barre scaricate da {df['timestamp'].iloc[0]} a {df['timestamp'].iloc[-1]}")
    return df

df_raw = load_market_data()
df_raw.head(3)


In [ ]:
# 4. Feature Engineering Stazionaria
FEATURE_COLUMNS = [
    "ret_close", "ret_high", "ret_low", "hl_spread",
    "norm_atr", "vol_vs_sma", "ret_volume",
    "dist_sma_50", "dist_sma_200", "dist_sma_50_200", "funding_rate"
]

def calculate_atr(df, period=14):
    high, low, close = df["high"], df["low"], df["close"]
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    return tr.rolling(window=period, min_periods=period).mean()

def extract_features(df):
    df = df.copy()
    df["ret_close"] = np.log(df["close"] / df["close"].shift(1))
    df["ret_high"] = np.log(df["high"] / df["high"].shift(1))
    df["ret_low"] = np.log(df["low"] / df["low"].shift(1))
    df["hl_spread"] = np.log(df["high"] / df["low"])
    df["norm_atr"] = calculate_atr(df, 14) / df["close"]
    vol_sma = df["volume"].rolling(20, min_periods=20).mean()
    df["vol_vs_sma"] = (df["volume"] - vol_sma) / (vol_sma + 1e-8)
    df["ret_volume"] = np.log((df["volume"] + 1.0) / (df["volume"].shift(1) + 1.0))
    sma50 = df["close"].rolling(50, min_periods=50).mean()
    sma200 = df["close"].rolling(200, min_periods=200).mean()
    df["dist_sma_50"] = (df["close"] - sma50) / (sma50 + 1e-8)
    df["dist_sma_200"] = (df["close"] - sma200) / (sma200 + 1e-8)
    df["dist_sma_50_200"] = (sma50 - sma200) / (sma200 + 1e-8)
    if "funding_rate" not in df.columns:
        df["funding_rate"] = 0.0
    df.dropna(subset=FEATURE_COLUMNS, inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

class FeatureScaler:
    def __init__(self):
        self.scaler = RobustScaler()
        self.clip_bounds = {}
        self.feature_names = FEATURE_COLUMNS
        
    def fit(self, df):
        X = df[FEATURE_COLUMNS].copy()
        for col in FEATURE_COLUMNS:
            self.clip_bounds[col] = (X[col].quantile(0.01), X[col].quantile(0.99))
            X[col] = X[col].clip(lower=self.clip_bounds[col][0], upper=self.clip_bounds[col][1])
        self.scaler.fit(X.values)
        return self

    def transform(self, df):
        X = df[FEATURE_COLUMNS].copy()
        for col in FEATURE_COLUMNS:
            X[col] = X[col].clip(lower=self.clip_bounds[col][0], upper=self.clip_bounds[col][1])
        return self.scaler.transform(X.values)

df_features = extract_features(df_raw)
print(f"Feature estratte: {len(df_features)} barre.")


In [ ]:
# 5. Triple Barrier Labeling
def apply_triple_barrier(
    df,
    horizon=6,
    tp_pct=0.020,
    sl_pct=0.014,
    use_dynamic_atr=False,
    atr_mult_tp=2.2,
    atr_mult_sl=1.4,
    min_atr_tp=0.015,
    min_atr_sl=0.010,
    taker_fee=0.0005
):
    n = len(df)
    labels = np.zeros(n, dtype=int)
    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    atrs = df["norm_atr"].values
    fee = 2 * taker_fee

    for t in range(n - horizon):
        p0 = closes[t]
        if use_dynamic_atr:
            target_tp = max(min_atr_tp, atrs[t] * atr_mult_tp)
            target_sl = max(min_atr_sl, atrs[t] * atr_mult_sl)
        else:
            target_tp = tp_pct
            target_sl = sl_pct

        long_tp = p0 * (1.0 + target_tp + fee)
        long_sl = p0 * (1.0 - target_sl - fee)
        short_tp = p0 * (1.0 - target_tp - fee)
        short_sl = p0 * (1.0 + target_sl + fee)

        l_hit, s_hit = None, None

        for k in range(1, horizon + 1):
            h, l = highs[t + k], lows[t + k]
            if h >= long_tp and l <= long_sl:
                break # Worst case conflitto intra-barra
            elif l <= long_sl:
                break
            elif h >= long_tp:
                l_hit = k
                break

        for k in range(1, horizon + 1):
            h, l = highs[t + k], lows[t + k]
            if l <= short_tp and h >= short_sl:
                break
            elif h >= short_sl:
                break
            elif l <= short_tp:
                s_hit = k
                break

        if l_hit is not None and s_hit is not None:
            labels[t] = 1 if l_hit < s_hit else (2 if s_hit < l_hit else 0)
        elif l_hit is not None:
            labels[t] = 1
        elif s_hit is not None:
            labels[t] = 2
        else:
            labels[t] = 0

    df_out = df.copy()
    df_out["label"] = labels
    return df_out.iloc[:-horizon].copy().reset_index(drop=True)


In [ ]:
# 6. TimeSeriesDataset e Split con Embargo
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, lookback=64):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.lookback = lookback

    def __len__(self):
        return len(self.X) - self.lookback + 1

    def __getitem__(self, idx):
        x_seq = self.X[idx : idx + self.lookback].t()
        label = self.y[idx + self.lookback - 1]
        return x_seq, label

def prepare_splits(df_labeled, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, embargo=12, bs=64, lookback=64):
    n = len(df_labeled)
    t_end = int(n * train_ratio)
    v_start = t_end + embargo
    v_end = v_start + int(n * val_ratio)
    te_start = v_end + embargo

    train_df = df_labeled.iloc[:t_end].copy().reset_index(drop=True)
    val_df = df_labeled.iloc[v_start:v_end].copy().reset_index(drop=True)
    test_df = df_labeled.iloc[te_start:].copy().reset_index(drop=True)

    scaler = FeatureScaler()
    train_X = scaler.fit(train_df).transform(train_df)
    val_X = scaler.transform(val_df)
    test_X = scaler.transform(test_df)

    train_loader = DataLoader(TimeSeriesDataset(train_X, train_df["label"].values, lookback), batch_size=bs, shuffle=True)
    val_loader = DataLoader(TimeSeriesDataset(val_X, val_df["label"].values, lookback), batch_size=bs, shuffle=False)
    test_loader = DataLoader(TimeSeriesDataset(test_X, test_df["label"].values, lookback), batch_size=bs, shuffle=False)

    return (train_df, val_df, test_df), (train_loader, val_loader, test_loader), scaler


In [ ]:
# 7. Causal Temporal Convolutional Network (TCN)
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.30):
        super().__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.norm1 = nn.GroupNorm(1, n_outputs)
        self.relu1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.norm2 = nn.GroupNorm(1, n_outputs)
        self.relu2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.norm1, self.relu1, self.drop1,
            self.conv2, self.chomp2, self.norm2, self.relu2, self.drop2
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.GELU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvolutionalNetwork(nn.Module):
    def __init__(self, num_features=11, channels=[32, 32, 48, 48, 64], kernel_size=3, dropout=0.30, num_classes=3):
        super().__init__()
        layers = []
        num_levels = len(channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_features if i == 0 else channels[i - 1]
            out_channels = channels[i]
            layers.append(
                TemporalBlock(
                    in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                    padding=(kernel_size - 1) * dilation_size, dropout=dropout
                )
            )
        self.network = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(channels[-1]),
            nn.Dropout(dropout),
            nn.Linear(channels[-1], num_classes)
        )

    def forward(self, x):
        y1 = self.network(x)
        h_T = y1[:, :, -1]  # Vettore causale finale al timestep corrente
        return self.classifier(h_T)

    def predict_probabilities(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            return F.softmax(logits, dim=-1).cpu()

def train_tcn_model(train_loader, val_loader, train_labels, lr=3e-4, dropout=0.30, weight_decay=0.01, epochs=35, patience=8):
    counts = np.bincount(train_labels, minlength=3)
    weights = len(train_labels) / (3.0 * np.maximum(counts, 1.0))
    weights_t = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model = TemporalConvolutionalNetwork(
        num_features=len(FEATURE_COLUMNS),
        channels=BASE_CONFIG["channels"],
        kernel_size=BASE_CONFIG["kernel_size"],
        dropout=dropout
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=weights_t)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_loss = float("inf")
    best_weights = None
    best_val_f1 = 0.0
    no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        v_loss, preds, targets = 0.0, [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                v_loss += criterion(out, y).item() * len(y)
                preds.extend(torch.argmax(out, dim=-1).cpu().numpy())
                targets.extend(y.cpu().numpy())
        v_loss /= len(val_loader.dataset)
        val_f1 = f1_score(targets, preds, average="macro", zero_division=0)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_val_f1 = val_f1
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_weights)
    return model.to(DEVICE), best_val_loss, best_val_f1

def predict_probabilities(model, loader):
    model.eval()
    all_p = []
    with torch.no_grad():
        for x, _ in loader:
            p = model.predict_probabilities(x.to(DEVICE))
            all_p.append(p.numpy())
    return np.concatenate(all_p, axis=0)


In [ ]:
# 8. Simulatore di Backtest Realistico
def run_simulation(
    test_bars,
    probs,
    prob_threshold=0.38,
    ratio_threshold=1.25,
    fixed_tp_pct=0.028,
    fixed_sl_pct=0.018,
    use_dynamic_atr=False,
    atr_mult_tp=2.2,
    atr_mult_sl=1.4,
    min_atr_tp=0.015,
    min_atr_sl=0.010,
    use_trailing_stop=False,
    breakeven_trigger_pct=0.012,
    be_tp_fraction=0.50,
    capital_allocation_pct=0.10,
    initial_capital=1000.0,
    taker_fee=0.0005,
    max_holding=6
):
    capital = initial_capital
    equity_curve = [capital]
    active_trade = None
    trades = []

    for t in range(len(test_bars) - 1):
        curr_bar = test_bars.iloc[t]
        next_bar = test_bars.iloc[t + 1]

        if active_trade is not None:
            active_trade["duration"] += 1
            h, l, c = curr_bar["high"], curr_bar["low"], curr_bar["close"]
            entry = active_trade["entry"]
            exit_p, reason = None, None

            if use_dynamic_atr:
                tp_dist = entry * active_trade["tp_target_pct"]
                sl_dist = entry * active_trade["sl_target_pct"]
                curr_tp = entry + tp_dist if active_trade["dir"] == "LONG" else entry - tp_dist
                curr_sl = entry - sl_dist if active_trade["dir"] == "LONG" else entry + sl_dist
            else:
                curr_tp = entry * (1.0 + fixed_tp_pct) if active_trade["dir"] == "LONG" else entry * (1.0 - fixed_tp_pct)
                curr_sl = entry * (1.0 - fixed_sl_pct) if active_trade["dir"] == "LONG" else entry * (1.0 + fixed_sl_pct)

            if use_trailing_stop:
                be_target_dist = max(breakeven_trigger_pct, active_trade["tp_target_pct"] * be_tp_fraction)
                if active_trade["dir"] == "LONG":
                    if h >= entry * (1.0 + be_target_dist):
                        active_trade["be_activated"] = True
                    if active_trade.get("be_activated", False):
                        curr_sl = max(curr_sl, entry * (1.0 + 2 * taker_fee))
                else:
                    if l <= entry * (1.0 - be_target_dist):
                        active_trade["be_activated"] = True
                    if active_trade.get("be_activated", False):
                        curr_sl = min(curr_sl, entry * (1.0 - 2 * taker_fee))

            if active_trade["dir"] == "LONG":
                if h >= curr_tp and l <= curr_sl:
                    exit_p, reason = curr_sl, "SL"
                elif l <= curr_sl:
                    exit_p, reason = curr_sl, "SL"
                elif h >= curr_tp:
                    exit_p, reason = curr_tp, "TP"
                elif active_trade["duration"] >= max_holding:
                    exit_p, reason = c, "TIME"
            else:
                if l <= curr_tp and h >= curr_sl:
                    exit_p, reason = curr_sl, "SL"
                elif h >= curr_sl:
                    exit_p, reason = curr_sl, "SL"
                elif l <= curr_tp:
                    exit_p, reason = curr_tp, "TP"
                elif active_trade["duration"] >= max_holding:
                    exit_p, reason = c, "TIME"

            if exit_p is not None:
                gross_ret = (exit_p - entry)/entry if active_trade["dir"] == "LONG" else (entry - exit_p)/entry
                net_ret = gross_ret - (2 * taker_fee)
                pnl = active_trade["size"] * net_ret
                capital += pnl
                trades.append({
                    "pnl": pnl, "net_ret": net_ret, "reason": reason, "dir": active_trade["dir"]
                })
                active_trade = None

        if active_trade is None and t < len(probs):
            p_neu, p_l, p_s = probs[t]
            signal = None
            if p_l >= prob_threshold and p_l >= ratio_threshold * p_s:
                signal = "LONG"
            elif p_s >= prob_threshold and p_s >= ratio_threshold * p_l:
                signal = "SHORT"

            if signal is not None:
                trade_size = max(50.0, capital * capital_allocation_pct)
                atr_val = curr_bar["norm_atr"]
                tp_trade_pct = max(min_atr_tp, atr_val * atr_mult_tp) if use_dynamic_atr else fixed_tp_pct
                sl_trade_pct = max(min_atr_sl, atr_val * atr_mult_sl) if use_dynamic_atr else fixed_sl_pct
                active_trade = {
                    "dir": signal,
                    "entry": next_bar["open"],
                    "duration": 0,
                    "size": trade_size,
                    "tp_target_pct": tp_trade_pct,
                    "sl_target_pct": sl_trade_pct,
                    "be_activated": False
                }

        equity_curve.append(capital)

    eq_series = pd.Series(equity_curve)
    drawdowns = (eq_series - eq_series.cummax()) / eq_series.cummax() * 100
    pnls = [t["pnl"] for t in trades]
    gains = [p for p in pnls if p > 0]
    losses = [abs(p) for p in pnls if p < 0]

    win_rate = (len(gains) / len(pnls) * 100) if pnls else 0.0
    profit_factor = (sum(gains) / sum(losses)) if sum(losses) > 0 else (99.0 if sum(gains) > 0 else 0.0)
    total_ret = ((capital - initial_capital) / initial_capital) * 100.0

    return {
        "total_return_pct": round(total_ret, 2),
        "max_drawdown_pct": round(abs(drawdowns.min()), 2),
        "win_rate_pct": round(win_rate, 1),
        "profit_factor": round(profit_factor, 2),
        "total_trades": len(trades),
        "equity_curve": equity_curve
    }


In [ ]:
# =====================================================================
# ESPERIMENTO 1: Grid Search su Validation Set (SOL/USDT)
# =====================================================================
print("--- AVVIO ESPERIMENTO 1: Tuning Iperparametri su Validation Set ---")
df_labeled_exp1 = apply_triple_barrier(
    df_features,
    horizon=6,
    tp_pct=0.035,
    sl_pct=0.022,
    use_dynamic_atr=False
)
(train_df, val_df, test_df), (train_loader, val_loader, test_loader), scaler_exp1 = prepare_splits(df_labeled_exp1)

grid_configs = [
    {"name": "C1 (Standard 3e-4, drop=0.30, wd=0.01)", "lr": 3e-4, "dropout": 0.30, "wd": 0.01},
    {"name": "C2 (High Reg 1.5e-4, drop=0.35, wd=0.02)", "lr": 1.5e-4, "dropout": 0.35, "wd": 0.02},
    {"name": "C3 (Moderate 5e-4, drop=0.25, wd=0.005)", "lr": 5e-4, "dropout": 0.25, "wd": 0.005}
]

best_exp1_model = None
best_exp1_f1 = -1.0
best_exp1_cfg = None

for cfg in grid_configs:
    print(f"Training {cfg['name']}...")
    model, v_loss, v_f1 = train_tcn_model(
        train_loader, val_loader, train_df["label"].values,
        lr=cfg["lr"], dropout=cfg["dropout"], weight_decay=cfg["wd"]
    )
    print(f"  --> Val Loss: {v_loss:.4f} | Val Macro F1: {v_f1:.4f}")
    if v_f1 > best_exp1_f1:
        best_exp1_f1 = v_f1
        best_exp1_model = model
        best_exp1_cfg = cfg

print(f"\nConfigurazione Vincitrice Exp 1: {best_exp1_cfg['name']} con Val F1 = {best_exp1_f1:.4f}")

probs_exp1 = predict_probabilities(best_exp1_model, test_loader)
test_bars = test_df.iloc[BASE_CONFIG["lookback_window"] - 1:].copy().reset_index(drop=True)

res_exp1 = run_simulation(
    test_bars, probs_exp1,
    prob_threshold=0.395,
    ratio_threshold=1.3,
    fixed_tp_pct=0.042,
    fixed_sl_pct=0.026,
    capital_allocation_pct=0.10,
    max_holding=6
)

print(f"Risultati Test Exp 1 (Baseline Tunata): Return={res_exp1['total_return_pct']}% | WinRate={res_exp1['win_rate_pct']}% | Trades={res_exp1['total_trades']} | DD={res_exp1['max_drawdown_pct']}%")


In [ ]:
# =====================================================================
# ESPERIMENTO 2: Dynamic ATR Volatility Barriers (SOL/USDT)
# =====================================================================
print("--- AVVIO ESPERIMENTO 2: Barriere Dinamiche Basate su ATR ---")
df_labeled_exp2 = apply_triple_barrier(
    df_features,
    horizon=6,
    use_dynamic_atr=True,
    atr_mult_tp=2.4,
    atr_mult_sl=1.5,
    min_atr_tp=0.035,
    min_atr_sl=0.022
)

(train_df2, val_df2, test_df2), (train_loader2, val_loader2, test_loader2), scaler_exp2 = prepare_splits(df_labeled_exp2)

print(f"Training TCN su label ATR Dinamiche con parametri vincenti ({best_exp1_cfg['name']})...")
model_exp2, v_loss2, v_f12 = train_tcn_model(
    train_loader2, val_loader2, train_df2["label"].values,
    lr=best_exp1_cfg["lr"], dropout=best_exp1_cfg["dropout"], weight_decay=best_exp1_cfg["wd"]
)
print(f"Modello Exp 2 completato -> Val Loss: {v_loss2:.4f} | Val Macro F1: {v_f12:.4f}")

probs_exp2 = predict_probabilities(model_exp2, test_loader2)
test_bars2 = test_df2.iloc[BASE_CONFIG["lookback_window"] - 1:].copy().reset_index(drop=True)

res_exp2 = run_simulation(
    test_bars2, probs_exp2,
    prob_threshold=0.395,
    ratio_threshold=1.3,
    use_dynamic_atr=True,
    atr_mult_tp=2.4,
    atr_mult_sl=1.5,
    min_atr_tp=0.035,
    min_atr_sl=0.022,
    capital_allocation_pct=0.10,
    max_holding=6
)

print(f"Risultati Test Exp 2 (Dynamic ATR): Return={res_exp2['total_return_pct']}% | WinRate={res_exp2['win_rate_pct']}% | Trades={res_exp2['total_trades']} | DD={res_exp2['max_drawdown_pct']}%")


In [ ]:
# =====================================================================
# ESPERIMENTO 3: Trailing Stop a Breakeven + Compound Sizing (15%)
# =====================================================================
print("--- AVVIO ESPERIMENTO 3: Advanced Execution (Breakeven Trailing + 15% Compound Sizing) ---")
champion_model = model_exp2 if res_exp2["total_return_pct"] >= res_exp1["total_return_pct"] else best_exp1_model
champion_probs = probs_exp2 if res_exp2["total_return_pct"] >= res_exp1["total_return_pct"] else probs_exp1
champion_bars = test_bars2 if res_exp2["total_return_pct"] >= res_exp1["total_return_pct"] else test_bars
champion_scaler = scaler_exp2 if res_exp2["total_return_pct"] >= res_exp1["total_return_pct"] else scaler_exp1

res_exp3 = run_simulation(
    champion_bars, champion_probs,
    prob_threshold=0.395,
    ratio_threshold=1.3,
    use_dynamic_atr=True if res_exp2["total_return_pct"] >= res_exp1["total_return_pct"] else False,
    fixed_tp_pct=0.042,
    fixed_sl_pct=0.026,
    atr_mult_tp=2.4,
    atr_mult_sl=1.5,
    min_atr_tp=0.035,
    min_atr_sl=0.022,
    max_holding=6,
    use_trailing_stop=True,
    breakeven_trigger_pct=0.025,
    be_tp_fraction=0.5,
    capital_allocation_pct=0.15
)

print(f"Risultati Test Exp 3 (Trailing + Compound): Return={res_exp3['total_return_pct']}% | WinRate={res_exp3['win_rate_pct']}% | Trades={res_exp3['total_trades']} | DD={res_exp3['max_drawdown_pct']}%")


In [ ]:
# =====================================================================
# 📊 CONFRONTO BENCHMARK MULTI-ESPERIMENTO & EXPORT CHAMPION (SOL/USDT)
# =====================================================================
summary_df = pd.DataFrame([
    {
        "Esperimento": "1. Baseline Tunata (Fisso 10%)",
        "Rendimento %": f"{res_exp1['total_return_pct']:+.2f}%",
        "Max Drawdown %": f"{res_exp1['max_drawdown_pct']:.2f}%",
        "Win Rate %": f"{res_exp1['win_rate_pct']:.1f}%",
        "Profit Factor": res_exp1["profit_factor"],
        "Trade Totali": res_exp1["total_trades"]
    },
    {
        "Esperimento": "2. Dynamic ATR Volatility",
        "Rendimento %": f"{res_exp2['total_return_pct']:+.2f}%",
        "Max Drawdown %": f"{res_exp2['max_drawdown_pct']:.2f}%",
        "Win Rate %": f"{res_exp2['win_rate_pct']:.1f}%",
        "Profit Factor": res_exp2["profit_factor"],
        "Trade Totali": res_exp2["total_trades"]
    },
    {
        "Esperimento": "3. Trailing Stop + Compound 15%",
        "Rendimento %": f"{res_exp3['total_return_pct']:+.2f}%",
        "Max Drawdown %": f"{res_exp3['max_drawdown_pct']:.2f}%",
        "Win Rate %": f"{res_exp3['win_rate_pct']:.1f}%",
        "Profit Factor": res_exp3["profit_factor"],
        "Trade Totali": res_exp3["total_trades"]
    }
])

print("\n" + "="*80)
print(f"TABELLA COMPARATIVA SOL/USDT SUL TEST SET OUT-OF-SAMPLE (POST-FEE):")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80 + "\n")

# Grafico delle Equity Curve
plt.figure(figsize=(13, 6))
plt.plot(res_exp1["equity_curve"], label=f"Exp 1: Baseline Tuned ({res_exp1['total_return_pct']:+.1f}%)", color="#0077b6", lw=1.5)
plt.plot(res_exp2["equity_curve"], label=f"Exp 2: Dynamic ATR ({res_exp2['total_return_pct']:+.1f}%)", color="#ffaa00", lw=1.5)
plt.plot(res_exp3["equity_curve"], label=f"Exp 3: Trailing Stop + 15% Compound ({res_exp3['total_return_pct']:+.1f}%)", color="#00a86b", lw=2.2)
plt.title("TCN Quantitative Trading Bot - Confronto Multi-Esperimento SOL/USDT (Post-Fee)")
plt.ylabel("Portfolio Equity ($)")
plt.xlabel("Barre Temporali (4H)")
plt.grid(True, alpha=0.3)
plt.legend(loc="upper left")
plt.tight_layout()
chart_path = f"/kaggle/working/comparison_sol_equity.png" if os.path.exists("/kaggle/working") else f"models/comparison_sol_equity.png"
plt.savefig(chart_path, dpi=200)
plt.show()

# Salvataggio Pesi e Scaler specifici per SOL
model_file_name = "tcn_sol_best_model.pth"
scaler_file_name = "scaler_sol.pkl"

model_save_path = os.path.join("/kaggle/working" if os.path.exists("/kaggle/working") else "models", model_file_name)
scaler_save_path = os.path.join("/kaggle/working" if os.path.exists("/kaggle/working") else "models", scaler_file_name)

torch.save(champion_model.state_dict(), model_save_path)
with open(scaler_save_path, "wb") as f:
    pickle.dump(champion_scaler, f)

# Salvataggio anche con nome generico per drop-in replacement immediato
generic_model_path = os.path.join("/kaggle/working" if os.path.exists("/kaggle/working") else "models", "tcn_best_model.pth")
generic_scaler_path = os.path.join("/kaggle/working" if os.path.exists("/kaggle/working") else "models", "scaler.pkl")
torch.save(champion_model.state_dict(), generic_model_path)
with open(generic_scaler_path, "wb") as f:
    pickle.dump(champion_scaler, f)

print(f"Modello SOL salvato in:  {model_save_path}")
print(f"Scaler SOL salvato in:   {scaler_save_path}")
print(f"Grafico salvato in:           {chart_path}")
print("\nScarica questi file per usarli con il bot!")
